# Automated Cost-Effectiveness Analysis Engine

This notebook demonstrates a transparent, reproducible cost-effectiveness analysis (CEA) workflow for public-health interventions. It uses simulated data for illustration only; replace the inputs with validated programme, epidemiological, and costing data before making decisions.

## Important methodological notes
- The original notebook generated different results on every run because it did not set a random seed.
- The original DALY formula mixed quantities with unclear units and was not a valid substitute for a formal DALY model. Here, DALYs averted are explicitly labelled as simulated inputs.
- The former `3 × GDP per capita` rule is not a universally valid WHO decision rule. The notebook therefore uses a configurable illustrative willingness-to-pay threshold and clearly labels it as an assumption.
- Costs are calculated for a cohort of 1,000 people, while the denominator is DALYs averted in that same cohort.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

RANDOM_SEED = 42
COHORT_SIZE = 1_000
WTP_THRESHOLD_USD_PER_DALY = 7_500  # Illustrative assumption
rng = np.random.default_rng(RANDOM_SEED)

## 1. Generate illustrative inputs

The simulated values below are not empirical estimates. They are included to demonstrate the workflow and should be replaced with auditable input data.

In [ ]:
interventions = [
    'HPV Vaccination',
    'Hypertension Screening',
    'TB Contact Tracing',
    'Malaria ITN',
]

rows = []
for intervention in interventions:
    cost_per_person = rng.uniform(10, 200)
    coverage = rng.uniform(0.20, 0.90)
    effectiveness = coverage * rng.beta(5, 2)
    dalys_averted_per_1000 = rng.uniform(0.5, 8.0)
    rows.append({
        'intervention': intervention,
        'cost_per_person_usd': cost_per_person,
        'coverage': coverage,
        'effectiveness': effectiveness,
        'dalys_averted_per_1000': dalys_averted_per_1000,
    })

df = pd.DataFrame(rows)
df['total_cost_usd'] = df['cost_per_person_usd'] * COHORT_SIZE / 1_000
df['cost_per_daly_averted_usd'] = (
    df['total_cost_usd'] / df['dalys_averted_per_1000']
)
df['cost_effective_at_threshold'] = (
    df['cost_per_daly_averted_usd'] <= WTP_THRESHOLD_USD_PER_DALY
)

## 2. Validate inputs

Basic validation prevents impossible values and division by zero from silently producing misleading results.

In [ ]:
assert df['intervention'].notna().all()
assert df['cost_per_person_usd'].ge(0).all()
assert df['coverage'].between(0, 1).all()
assert df['effectiveness'].between(0, 1).all()
assert df['dalys_averted_per_1000'].gt(0).all()
assert df['cost_per_daly_averted_usd'].notna().all()

## 3. Rank interventions

Lower cost per DALY averted indicates better cost-effectiveness under this simplified framework. The result is not, by itself, a recommendation: affordability, equity, feasibility, uncertainty, and comparator choice also matter.

In [ ]:
ranking = (
    df.sort_values('cost_per_daly_averted_usd')
      .reset_index(drop=True)
)

display(
    ranking[[
        'intervention',
        'cost_per_person_usd',
        'coverage',
        'dalys_averted_per_1000',
        'cost_per_daly_averted_usd',
        'cost_effective_at_threshold',
    ]].style.format({
        'cost_per_person_usd': '${:,.2f}',
        'coverage': '{:.1%}',
        'dalys_averted_per_1000': '{:.2f}',
        'cost_per_daly_averted_usd': '${:,.2f}',
    })
)

## 4. Interpretation

The threshold below is a user-supplied illustrative assumption, not a global standard. For an applied analysis, define the comparator, perspective, time horizon, discount rate, currency year, costing method, and decision threshold in advance.

In [ ]:
print(f'Illustrative threshold: ${WTP_THRESHOLD_USD_PER_DALY:,.0f} per DALY averted')
print(f'Random seed: {RANDOM_SEED}')
print('\nRanked results:')
print(
    ranking[['intervention', 'cost_per_daly_averted_usd', 'cost_effective_at_threshold']]
    .to_string(index=False, formatters={
        'cost_per_daly_averted_usd': '${:,.2f}'.format
    })
)

## Limitations and next steps

- Replace simulated values with sourced intervention costs and epidemiological outcomes.
- Model DALYs using age-specific mortality, morbidity, disability weights, and time horizons where appropriate.
- Specify a healthcare-system, payer, provider, or societal perspective.
- Add discounting, uncertainty analysis, sensitivity analysis, and budget impact analysis.
- Compare incremental cost-effectiveness ratios against a clearly justified comparator.
- Report currency, price year, exchange-rate or purchasing-power adjustment, and data sources.